In [1]:
import pandas as pd
import numpy as np

In [ ]:
# 1. Subscription per Month
global_monthly_signups = pd.Series(dtype=float)

# 2. Top 5 Countries (& its cities) subscribers
global_country_counts = pd.Series(dtype=float)

# 3. B2B or B2C?
global_b2c_count = 0
global_b2b_count = 0
free_domains = {'gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'icloud.com', 'aol.com'}

### Load large data through chunking

In [3]:
chunk_iterator = pd.read_csv('data/customers-2000000.csv', chunksize=100000)

In [4]:
for i, chunk in enumerate(chunk_iterator, 1):
  print(f"processing chunk {i}")

  ## 1. Subscription per Month
  if "Subscription Date" in chunk.columns:
    chunk['Subscription Date'] = pd.to_datetime(chunk['Subscription Date'], errors='coerce')
    local_monthly_counts = chunk['Subscription Date'].dt.to_period('M').value_counts()
    global_monthly_signups = global_monthly_signups.add(local_monthly_counts, fill_value=0)

  ## 2. Top 5 countries
  local_country_counts = chunk['Country'].value_counts()
  global_country_counts = global_country_counts.add(local_country_counts, fill_value=0)

  ## 3. B2B or B2C?
  if 'Email' in chunk.columns:
    domains = chunk['Email'].str.split('@').str[1].str.lower()
    is_b2c = domains.isin(free_domains)
    
    # Tally up the booleans (True evaluates to 1, False to 0)
    global_b2c_count += is_b2c.sum()
    global_b2b_count += (~is_b2c).sum()

processing chunk 1
processing chunk 2
processing chunk 3
processing chunk 4
processing chunk 5
processing chunk 6
processing chunk 7
processing chunk 8
processing chunk 9
processing chunk 10
processing chunk 11
processing chunk 12
processing chunk 13
processing chunk 14
processing chunk 15
processing chunk 16
processing chunk 17
processing chunk 18
processing chunk 19
processing chunk 20


### 1. Subscribers per Month and Growth

In [5]:
global_monthly_signups.sort_index(inplace=True)
    
mom_growth = global_monthly_signups.pct_change().fillna(0) * 100

velocity_df = pd.DataFrame({
    'Total Signups': global_monthly_signups.astype(int),
    'MoM Growth (%)': mom_growth.round(2)
})

print(velocity_df.tail(6))

                   Total Signups  MoM Growth (%)
Subscription Date                               
2021-12                    70130            2.80
2022-01                    70093           -0.05
2022-02                    63671           -9.16
2022-03                    70249           10.33
2022-04                    68472           -2.53
2022-05                    66078           -3.50


### 2. Top 5 Subscribers Countries

In [6]:
print(global_country_counts.sort_values(ascending=False).head(5).astype(int))

Country
Korea       16240
Congo       16208
Jordan       8428
Vietnam      8388
Suriname     8375
dtype: int64


### 3. B2B or B2C?

In [7]:
total_users = global_b2b_count + global_b2c_count
if total_users > 0:
    print(f"B2C (Consumer): {(global_b2c_count / total_users * 100):.2f}%")
    print(f"B2B (Corporate): {(global_b2b_count / total_users * 100):.2f}%")

B2C (Consumer): 0.00%
B2B (Corporate): 100.00%
